# Expectation-Maximization (EM) Algorithm on Galton Family Heights

This notebook implements a **two‑component Gaussian mixture model** without labels.  
We use the famous GaltonFamilies dataset and try to separate the heights into two clusters: *children* and *fathers*, **without** using the family identifiers.  
The EM algorithm iteratively assigns soft membership probabilities (E‑step) and updates the parameters of the two Gaussians (M‑step).

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup and Data Loading

In [3]:
import csv
import math
import os

# Get dataset
FILE = "/content/drive/MyDrive/GaltonFamilies.csv"

In [4]:
# === Load The Dataset ===
heights = []
seen_families = set()

with open(FILE, "r") as file:
    reader = csv.reader(file)
    next(reader)  # skip header

    for row in reader:
        family_id = row[1]
        father_height = float(row[2])
        child_height = float(row[8])

        # Add child's height to our dataset
        heights.append(child_height)

        # Add father only once per family to avoid duplicates
        if family_id not in seen_families:
            heights.append(father_height)
            seen_families.add(family_id)

N = len(heights)
print(f"Total individuals in the combined dataset: {N}")
print(f"First 5 heights: {heights[:5]}")

Total individuals in the combined dataset: 1139
First 5 heights: [73.2, 78.5, 69.2, 69.0, 69.0]


## 2. EM Algorithm Implementation

In [5]:
# Gaussian probability density function
def calculate_gaussian_prob(x, mean, variance):
    coefficient = 1.0 / math.sqrt(2.0 * math.pi * variance)
    exponent = math.exp(-0.5 * ((x - mean) ** 2) / variance)
    return coefficient * exponent

In [6]:
# === Initialization ===
# Guesses for the two clusters
mu1 = 64.0
mu2 = 72.0
var1 = 10.0
var2 = 10.0
pi1 = 0.5
pi2 = 0.5

# Responsibility vectors (soft assignments)
weights1 = [0.0] * N
weights2 = [0.0] * N

# Convergence settings
tolerance = 1e-8
max_iterations = 100
previous_log_likelihood = None

print("Iteration | mu1 (Children) | mu2 (Fathers) | var1 | var2 | pi1 | pi2 | Log-Likelihood")
print("-" * 85)

Iteration | mu1 (Children) | mu2 (Fathers) | var1 | var2 | pi1 | pi2 | Log-Likelihood
-------------------------------------------------------------------------------------


In [7]:
# === The EM Loop ===
for iteration in range(max_iterations):
    log_likelihood = 0.0

    # --- E-STEP ---
    for i in range(N):
        x = heights[i]

        prob_is_group1 = pi1 * calculate_gaussian_prob(x, mu1, var1)
        prob_is_group2 = pi2 * calculate_gaussian_prob(x, mu2, var2)
        total_prob = prob_is_group1 + prob_is_group2

        log_likelihood += math.log(total_prob)

        weights1[i] = prob_is_group1 / total_prob
        weights2[i] = prob_is_group2 / total_prob

    # Print current state
    print(f"{iteration:9} | {mu1:14.2f} | {mu2:10.2f} | {var1:4.2f} | {var2:4.2f} | {pi1:3.2f} | {pi2:3.2f} | {log_likelihood:.2f}")

    # Convergence check
    if previous_log_likelihood is not None:
        change = abs(log_likelihood - previous_log_likelihood)
        if change < tolerance:
            print(f"\n Converged after {iteration} iterations.")
            break
    previous_log_likelihood = log_likelihood

    # --- M-STEP ---
    N1 = sum(weights1)
    N2 = sum(weights2)

    pi1 = N1 / N
    pi2 = N2 / N

    mu1 = sum(weights1[i] * heights[i] for i in range(N)) / N1
    mu2 = sum(weights2[i] * heights[i] for i in range(N)) / N2

    var1 = sum(weights1[i] * ((heights[i] - mu1) ** 2) for i in range(N)) / N1
    var2 = sum(weights2[i] * ((heights[i] - mu2) ** 2) for i in range(N)) / N2

print("\nFinal parameters:")
print(f"mu1 = {mu1:.2f}, var1 = {var1:.2f}, pi1 = {pi1:.3f}")
print(f"mu2 = {mu2:.2f}, var2 = {var2:.2f}, pi2 = {pi2:.3f}")

        0 |          64.00 |      72.00 | 10.00 | 10.00 | 0.50 | 0.50 | -3238.13
        1 |          64.99 |      70.08 | 7.11 | 5.41 | 0.56 | 0.44 | -3048.92
        2 |          64.97 |      70.07 | 7.15 | 5.23 | 0.56 | 0.44 | -3048.61
        3 |          64.94 |      70.07 | 7.09 | 5.16 | 0.56 | 0.44 | -3048.51
        4 |          64.93 |      70.07 | 7.03 | 5.13 | 0.56 | 0.44 | -3048.43
        5 |          64.91 |      70.08 | 6.97 | 5.10 | 0.56 | 0.44 | -3048.37
        6 |          64.90 |      70.07 | 6.91 | 5.09 | 0.55 | 0.45 | -3048.32
        7 |          64.88 |      70.07 | 6.87 | 5.08 | 0.55 | 0.45 | -3048.28
        8 |          64.87 |      70.07 | 6.83 | 5.08 | 0.55 | 0.45 | -3048.24
        9 |          64.86 |      70.06 | 6.79 | 5.07 | 0.55 | 0.45 | -3048.20
       10 |          64.85 |      70.06 | 6.76 | 5.08 | 0.55 | 0.45 | -3048.17
       11 |          64.84 |      70.05 | 6.73 | 5.08 | 0.55 | 0.45 | -3048.13
       12 |          64.83 |      70.05 | 6.70 | 5

## 3. Live Classification Test

Using the learned parameters, we can classify a new height without knowing whether it comes from a child or a father.

In [8]:
test_height = 69.0

p_child = pi1 * calculate_gaussian_prob(test_height, mu1, var1)
p_father = pi2 * calculate_gaussian_prob(test_height, mu2, var2)
p_total = p_child + p_father

final_prob_child = p_child / p_total
final_prob_father = p_father / p_total

print(f"Test Height Evaluated: {test_height} inches")
print(f"Probability it is a Child  (Group 1): {final_prob_child * 100:.2f}%")
print(f"Probability it is a Father (Group 2): {final_prob_father * 100:.2f}%")

Test Height Evaluated: 69.0 inches
Probability it is a Child  (Group 1): 11.52%
Probability it is a Father (Group 2): 88.48%
